[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LegalIntermediaSL/Physics/blob/main/Simulaciones_Jupyter/Proyectos_de_Supercomputacion/Mecanica_Clasica_HPC_2.ipynb)



# Supercomputación: Mecanica Clasica HPC 2

Simulación computacional de altísimo rendimiento (HPC) orientada a sistemas ultracomplejos.

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt

def lbm_cylinder():
    maxIter = 1000; Re = 10.0; nx = 100; ny = 40; cx, cy = nx//4, ny//2; r = ny//9; u0 = 0.1
    nulb = u0*r/Re; omega = 1 / (3*nulb + 0.5)
    v = np.array([[1,1],[1,0],[1,-1],[0,1],[0,0],[0,-1],[-1,1],[-1,0],[-1,-1]])
    t = np.array([1/36, 1/9, 1/36, 1/9, 4/9, 1/9, 1/36, 1/9, 1/36])
    obstacle = np.fromfunction(lambda x,y: (x-cx)**2+(y-cy)**2 < r**2, (nx,ny))
    vel = np.fromfunction(lambda d,x,y: (1-d)*u0, (2,nx,ny)); feq = np.zeros((9,nx,ny)); fin = np.zeros((9,nx,ny))
    
    for i, cx2 in enumerate(v):
        fin[i] = t[i] * (1 + 3*(cx2[0]*vel[0] + cx2[1]*vel[1]))
        
    for time in range(maxIter):
        fin[:, -1, :] = fin[:, -2, :] 
        rho = np.sum(fin, axis=0)
        u = np.dot(v.T, fin.transpose((1,0,2))) / rho
        for i, cx2 in enumerate(v):
            feq[i] = rho * t[i] * (1 + 3*(cx2[0]*u[0]+cx2[1]*u[1]) + 9*(cx2[0]*u[0]+cx2[1]*u[1])**2/2 - 3*(u[0]**2+u[1]**2)/2)
        fout = fin - omega * (fin - feq)
        for i in range(9): fout[i, obstacle] = fin[8-i, obstacle]
        for i, cx2 in enumerate(v): fin[i] = np.roll(np.roll(fout[i], cx2[0], axis=0), cx2[1], axis=1)
        
    plt.imshow(np.sqrt(u[0]**2+u[1]**2).T, cmap='viridis')
    plt.savefig('lbm_velocity.png')

if __name__ == '__main__': lbm_cylinder()
